# 🇹🇳 Chat with the Tunisian model (testers' notebook)

Loads the **already-trained** LoRA adapter and lets you chat. No training, fast to start.

## Setup
1. **+ Add Data** → add the LoRA dataset the owner shared (the `tunisian_lora` folder,
   uploaded as a Kaggle Dataset — it contains `adapter_config.json`).
2. Right panel: **Accelerator = GPU T4**, **Internet = ON** (to download the base model).
3. **Run All**, then use the chat cell at the bottom.


## 1. Install

In [ ]:
%%capture
!pip install -q -U "transformers<5" "peft>=0.12" "bitsandbytes>=0.43" accelerate


## 2. Config — base model MUST match what was trained

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # change if the owner trained on 7B/8B
SYSTEM = ('Enti assistant tunsi (service client w 7adith 3am). Jaweb DIMA bel derja tounsiya '
          'bel arabizi (7ourouf latiniya w arqam), b tari9a tabi3iya w 9sira. Ken el user yekteb '
          'bel 3arbi wala faransi wala anglais, efhem w jaweb bel arabizi.')
import glob, os
def find_adapter():
    h = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
    if not h: raise FileNotFoundError('LoRA adapter not found — add the shared tunisian_lora dataset')
    return os.path.dirname(h[0])
ADAPTER = find_adapter(); print('adapter:', ADAPTER)


## 3. Load base model (4-bit) + the trained adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                            device_map='auto', torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print('ready ✅')


## 4. Chat helper

In [ ]:
def generate(user_msg, system=SYSTEM, max_new_tokens=200, temperature=0.7):
    msgs = [{'role':'system','content':system},{'role':'user','content':user_msg}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
            max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


## 5. Quick examples

In [ ]:
for m in ['3aslema chna7welek?','قداش تمن التوصيل لتونس؟','give me a dinner idea','9olli nokta']:
    print('🧑', m); print('🤖', generate(m), '\n')


## 6. 💬 Live chat — run this cell and type. Type `quit` to stop.

In [ ]:
print('Chat bel tounsi! ekteb "quit" bch to5rej.\n')
while True:
    try:
        msg = input('🧑 enti: ')
    except (EOFError, KeyboardInterrupt):
        break
    if msg.strip().lower() in ('quit','exit','q',''): print('بسلامة 👋'); break
    print('🤖', generate(msg), '\n')
